# Topic Modelling

In text mining, we often have collections of documents, such as blog posts or news articles, that we’d like to divide into natural groups so that we can understand them separately. Topic modeling is a method for unsupervised classification of such documents, similar to clustering on numeric data, which finds natural groups of items even when we’re not sure what we’re looking for.
Latent Dirichlet allocation (LDA) is a particularly popular method for fitting a topic model. It treats each document as a mixture of topics, and each topic as a mixture of words. This allows documents to “overlap” each other in terms of content, rather than being separated into discrete groups, in a way that mirrors typical use of natural language.


In [ ]:
%%capture
%pip install stop-words
%pip install gensim

In [ ]:
# We will start by taking a few sentences and treating them as documents
from nltk.tokenize import RegexpTokenizer
from stop_words import get_stop_words
from nltk.stem.porter import PorterStemmer
from gensim import corpora, models
import gensim

tokenizer = RegexpTokenizer(r'\w+')

# create English stop words list
en_stop = get_stop_words('en')

# Create p_stemmer of class PorterStemmer
p_stemmer = PorterStemmer()
    
# create sample documents
doc_a = "Brocolli is good to eat. My brother likes to eat good brocolli, but not my mother."
doc_b = "My mother spends a lot of time driving my brother around to baseball practice."
doc_c = "Some health experts suggest that driving may cause increased tension and blood pressure."
doc_d = "I often feel pressure to perform well at school, but my mother never seems to drive my brother to do better."
doc_e = "Health professionals say that brocolli is good for your health." 

# compile sample documents into a list
doc_set = [doc_a, doc_b, doc_c, doc_d, doc_e]



In [ ]:
# list for tokenized documents in loop
texts = []

# We will clean and tokenize our “documents using stemming
# loop through document list
for doc in doc_set:
    # clean and tokenize document string
    raw = doc.lower()
    tokens = tokenizer.tokenize(raw)

    # remove stop words from tokens
    stopped_tokens = [token for token in tokens if not token in en_stop]
    
    # stem tokens
    stemmed_tokens = [p_stemmer.stem(token) for token in stopped_tokens]
    
    # add tokens to list
    texts.append(stemmed_tokens)
print(texts)

During this cleaning process we introduce a new cleaning step called stemming
(bringing words to their original stem).

We will also be introducing lemmatization later in this laboratory.

Stemming and Lemmatization both generate the root form of the inflected words. The difference is that stem might not be an actual word. Lemma on the other hand is an actual word of target language.

Stemming follows an algorithm with steps to perform on the words which makes it faster. 

Lemmatization, however, uses both WordNet corpus and stop words corpus which makes it slower than stemming. You also had to define parts-of-speech to obtain the correct lemma.


We create a dictionary containing all the words in our documents

In [ ]:
# turn our tokenized documents into a id <-> term dictionary
dictionary = corpora.Dictionary(texts)
print(dictionary[0], dictionary[1], dictionary[2], dictionary[3], dictionary[4], dictionary[5])

In [ ]:
# convert tokenized documents into a document-term matrix (each row consisting of (word_id_in_dictionary, word_frequency_in_current_document))
corpus = [dictionary.doc2bow(text) for text in texts]

print(corpus[0]) # ['brocolli', 'good', 'eat', 'brother', 'like', 'eat', 'good', 'brocolli', 'mother']
print(corpus[1]) # ['mother', 'spend', 'lot', 'time', 'drive', 'brother', 'around', 'basebal', 'practice']

Finally we create our LDA model and specify the number of topics that we want/expect and for how many passes LDA should run.

In [ ]:
# generate LDA model
ldamodel = gensim.models.ldamodel.LdaModel(corpus, num_topics=2, id2word = dictionary, passes=20)


We can now access the model’s identified topics and the words that they are comprised of

In [ ]:
print(ldamodel.print_topics(num_topics=2, num_words=4))

### Wikipedia
Next up we are going to also introduce the wikipedia python module which we can use to download wikipedia pages.

In [ ]:
%%capture
%pip install wikipedia

In [ ]:
import wikipedia
import nltk

nltk.download('stopwords')
en_stop = set(nltk.corpus.stopwords.words('english'))

religion = wikipedia.page("Religion")
artificial_intelligence = wikipedia.page("Artificial Intelligence")
mona_lisa = wikipedia.page("Mona_Lisaa")
eiffel_tower = wikipedia.page("Eiffel Tower")


In [ ]:
corpus = [religion.content, artificial_intelligence.content, mona_lisa.content, eiffel_tower.content]

In [ ]:
print(mona_lisa.content)

In [ ]:
# Lemmatization

import re
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')

# We are going to create a function for text preprocessing since we are going to be analyzing multiple texts and will want to reuse the code.
stemmer = WordNetLemmatizer()

def preprocess_text(document):
        # Remove all the special characters
        document = re.sub(r'\W', ' ', str(document))

        # remove all single characters
        document = re.sub(r'\s+[a-zA-Z]\s+', ' ', document)

        # Remove single characters from the start
        document = re.sub(r'\^[a-zA-Z]\s+', ' ', document)

        # Substituting multiple spaces with single space
        document = re.sub(r'\s+', ' ', document, flags=re.I)

        # Removing prefixed 'b'
        document = re.sub(r'^b\s+', '', document)

        # Converting to Lowercase
        document = document.lower()

        # Lemmatization
        tokens = document.split()
        tokens = [stemmer.lemmatize(word) for word in tokens]
        tokens = [word for word in tokens if word not in en_stop]
        tokens = [word for word in tokens if len(word)  > 5]

        return tokens


In [ ]:
processed_data = [];
for doc in corpus:
    tokens = preprocess_text(doc)
    processed_data.append(tokens)

print(corpus[0][:205]) # Original text
print(processed_data[0][:15]) # Processed tokens


In this text preprocessing we’ve added the lemmatization step instead of the stemming one (remember the difference discussed earlier)

We preprocess all of the documents and add them to the processed_data list of documents


In [ ]:
%%capture
%pip install gensim==4.1

In [ ]:
from gensim import corpora

gensim_dictionary = corpora.Dictionary(processed_data)
gensim_corpus = [gensim_dictionary.doc2bow(token, allow_update=True) for token in processed_data]


(Optionally) - we can save our dictionary, corpus and model if we want to load them back later on (and not run it again risking to obtain different values next time we run it)


In [ ]:
import pickle

pickle.dump(gensim_corpus, open('gensim_corpus_corpus.pkl', 'wb'))
gensim_dictionary.save('gensim_dictionary.gensim')


In [ ]:
import gensim

lda_model = gensim.models.ldamodel.LdaModel(gensim_corpus, num_topics=4, id2word=gensim_dictionary, passes=150)
lda_model.save('gensim_model.gensim')


In [ ]:
topics = lda_model.print_topics(num_words=5)
for topic in topics:
    print(topic[1], '\n')


In [ ]:
lda_model_tst = gensim.models.ldamodel.LdaModel(gensim_corpus, num_topics=8, id2word=gensim_dictionary, passes=15)
topics = lda_model_tst.print_topics(num_words=5)
for topic in topics:
    print(topic[1], '\n')


Let’s test our LDA model that was trained on our previous 4 documents (resulting in 3 or 4 topics) on some sentences written by us. We pretend a sentence is a document.

In [ ]:
test_doc = 'There are many passages about Christ in The Bible'

In [ ]:
# To which topic does the sentence above resemble the most?
test_doc = preprocess_text(test_doc)
bow_test_doc = gensim_dictionary.doc2bow(test_doc)

print(lda_model.get_document_topics(bow_test_doc))


In [ ]:
test_doc = 'Da Vinci created impressive paintings'
test_doc = preprocess_text(test_doc)
bow_test_doc = gensim_dictionary.doc2bow(test_doc)

print(lda_model.get_document_topics(bow_test_doc))


In [ ]:
test_doc = 'Neural networks are taking over the scene of image recognition'
test_doc = preprocess_text(test_doc)
bow_test_doc = gensim_dictionary.doc2bow(test_doc)

print(lda_model.get_document_topics(bow_test_doc))


There are some metrics that we can print which give us an idea of how well the model is doing (in comparison to models for other documents). These metrics are perplexity and coherence.
You can think of the perplexity metric as measuring how probable some new unseen data is given the model that was learned earlier. That is to say, how well does the model represent or reproduce the statistics of the held-out data.
In order to decide the optimum number of topics to be extracted using LDA, topic coherence score is always used to measure how well the topics are extracted

In [ ]:
print('Perplexity:', lda_model.log_perplexity(gensim_corpus))

In [ ]:
from gensim.models import CoherenceModel

coherence_score_lda = CoherenceModel(model=lda_model, texts=processed_data, dictionary=gensim_dictionary, coherence='c_v')
coherence_score = coherence_score_lda.get_coherence()

print('Coherence Score:', coherence_score)


In [ ]:
%%capture
%pip install pyLDAvis

### Visualizing topics

In [ ]:
# If we want to load previously saved model
# gensim_dictionary = gensim.corpora.Dictionary.load('gensim_dictionary.gensim')
# gensim_corpus = pickle.load(open('gensim_corpus_corpus.pkl', 'rb'))
# lda_model = gensim.models.ldamodel.LdaModel.load('gensim_model.gensim')


import pyLDAvis.gensim_models

lda_visualization = pyLDAvis.gensim_models.prepare(lda_model, gensim_corpus, gensim_dictionary, sort_topics=False)
pyLDAvis.display(lda_visualization)


pyLDAvis is quite powerful of a tool and provides multiple visualizations within one interactive interface.


We can see the most relevant terms per topic and the most salient terms overall.
We can also play around with the relevance metric to obtain terms that are more or less relevant to a given topic.


The visualization is interactive, we can interact with what is being displayed and we can choose for instance what topic we want to further analyze. 


### LSA - Latent Semantic Analysis

As an extra example we also have an LSA model available in our gensim module which may give more coherent topics at the cost of the separation quality between topics.

In [ ]:
from gensim.models import LsiModel

lsi_model = LsiModel(gensim_corpus, num_topics=4, id2word=gensim_dictionary)
topics = lsi_model.print_topics(num_words=10)
for topic in topics:
    print(topic[1])


### Exercise 1
Choose 8 Wikipedia articles of your choice.


In [ ]:
article_ids = [
    22934, # Probability
    27619007, # Mossad
    63478112, # 1975 Banqiao Dam Failure
    147809, # Mochi
    28661, # Defamation
    344287, # Mercury Poisoning
    4279208, # Peer-to-peer file sharing
    1558354, # Demographics of Europe
]

articles = [wikipedia.page(pageid=id) for id in article_ids]
article_content = [article.content for article in articles]


1. Clean the data using **stemming** and print the resulting topics when applying LDA with num_topics = 2.


In [ ]:
stemmer = PorterStemmer()
tokenizer = RegexpTokenizer(r'\w+')
en_stop = get_stop_words('en')

wiki_texts = []
for content in article_content:
    raw = content.lower()
    words = tokenizer.tokenize(raw)
    
    stemmed_words = [stemmer.stem(word) for word in words if word not in en_stop]
    
    wiki_texts.append(stemmed_words)

In [ ]:
wiki_dict = corpora.Dictionary(wiki_texts)
wiki_corpus = [wiki_dict.doc2bow(text) for text in wiki_texts]

In [ ]:
ldamodel = gensim.models.ldamodel.LdaModel(wiki_corpus, num_topics=2, id2word=wiki_dict, passes=20)

print(ldamodel.print_topics(num_topics=2, num_words=4))


2. Clean the data using **lemmatization** and print the resulting topics when applying LDA with num_topics = 2.


In [ ]:
stemmer = WordNetLemmatizer()

wiki_texts = []
for content in article_content:
    lemmatized_words = preprocess_text(content)
    wiki_texts.append(lemmatized_words)

In [ ]:
wiki_dict = corpora.Dictionary(wiki_texts)
wiki_corpus = [wiki_dict.doc2bow(text) for text in wiki_texts]

In [ ]:
lda_model = gensim.models.ldamodel.LdaModel(wiki_corpus, num_topics=8, id2word=wiki_dict, passes=70)

topics = lda_model.print_topics(num_topics=8, num_words=4)
for topic in topics:
    print(topic[1])


3. Print the coherence and perplexity score of your lemmatization LDA (n=2) model.
Tune your LDA model hyperparameters (num_topics, num_passes, etc.) until you obtain as good of a coherence score as possible.

    Print the **coherence score, perplexity score and the topic separation** of your **new best model**.


In [ ]:
print('Perplexity:', lda_model.log_perplexity(wiki_corpus))

In [ ]:
from gensim.models import CoherenceModel

coherence_score_lda = CoherenceModel(model=lda_model, texts=wiki_texts, dictionary=wiki_dict, coherence='c_v')
coherence_score = coherence_score_lda.get_coherence()

print('Coherence Score:', coherence_score)


4. Apply LSA on your lemmatization-based cleaned articles and print the resulting topics using (num_topics = 2, num_topics = 4 and num_topics = 8)

In [ ]:
for num_topics in [2, 4, 8]:
    lsi_model = LsiModel(wiki_corpus, num_topics=num_topics, id2word=wiki_dict)
    topics = lsi_model.print_topics(num_words=6)
    print(f'LSA for number of topics: {num_topics}')
    for topic in topics:
        print(topic[1])

### Topic Modelling on books

In [ ]:
# We will try to apply topic modelling on the science books from last laboratories
import pandas as pd
import gutenbergpy.textget
from tidytext import unnest_tokens
import nltk
nltk.download('punkt')

raw_book = gutenbergpy.textget.get_text_by_id(37729)
galileo_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

raw_book = gutenbergpy.textget.get_text_by_id(14725)
huygens_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

raw_book = gutenbergpy.textget.get_text_by_id(13476)
tesla_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

raw_book = gutenbergpy.textget.get_text_by_id(30155)
einstein_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

corpus = [galileo_text, huygens_text, tesla_text, einstein_text]
processed_data = [];
for doc in corpus:
    tokens = preprocess_text(doc)
    processed_data.append(tokens)

gensim_dictionary = corpora.Dictionary(processed_data)
gensim_corpus = [gensim_dictionary.doc2bow(token, allow_update=True) for token in processed_data]

In [ ]:
lda_model = gensim.models.ldamodel.LdaModel(gensim_corpus, num_topics=4, id2word=gensim_dictionary, passes=255)

topics = lda_model.print_topics(num_words=15)
for topic in topics:
    print(topic, '\n')


In [ ]:
# See which book resembles to which topic

print(lda_model[gensim_corpus[0]])
print(lda_model[gensim_corpus[1]])
print(lda_model[gensim_corpus[2]])
print(lda_model[gensim_corpus[3]])


In [ ]:
# Let's see where some partial text from Tesla's text seems to belong to
tesla_text_crumbs = tesla_text[1000:5000]

tesla_text_crumbs = preprocess_text(tesla_text_crumbs)
tesla_text_crumbs = gensim_dictionary.doc2bow(tesla_text_crumbs)

print(lda_model.get_document_topics(tesla_text_crumbs))


### Jane Austen

In [ ]:
import pandas as pd
import gutenbergpy.textget
from tidytext import unnest_tokens
import nltk
nltk.download('punkt')

raw_book = gutenbergpy.textget.get_text_by_id(161)
sense_sensibility_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

raw_book = gutenbergpy.textget.get_text_by_id(1342)
pride_prejudice_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

raw_book = gutenbergpy.textget.get_text_by_id(158)
emma_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

raw_book = gutenbergpy.textget.get_text_by_id(105)
persuasion_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

corpus = [sense_sensibility_text, pride_prejudice_text, emma_text, persuasion_text]

processed_data = [];
for doc in corpus:
    tokens = preprocess_text(doc)
    processed_data.append(tokens)
    
gensim_dictionary = corpora.Dictionary(processed_data)
gensim_corpus = [gensim_dictionary.doc2bow(token, allow_update=True) for token in processed_data]

lda_model = gensim.models.ldamodel.LdaModel(gensim_corpus, num_topics=4, id2word=gensim_dictionary, passes=255)

topics = lda_model.print_topics(num_words=15)
for topic in topics:
    print(topic, '\n')


In [ ]:
print(lda_model[gensim_corpus[0]])
print(lda_model[gensim_corpus[1]])
print(lda_model[gensim_corpus[2]])
print(lda_model[gensim_corpus[3]])

## Dostoyevsky

In [ ]:
# Let's add some books from Dostoyevsky to our books corpus
raw_book = gutenbergpy.textget.get_text_by_id(2554)
crime_and_punishment_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

raw_book = gutenbergpy.textget.get_text_by_id(600)
notes_from_underground_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")

raw_book = gutenbergpy.textget.get_text_by_id(8117)
the_possessed_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")


In [ ]:
# Does the LDA model split the topics as being the 2 authors?
authors_corpus = [sense_sensibility_text, pride_prejudice_text, emma_text, persuasion_text,
                  crime_and_punishment_text, notes_from_underground_text, the_possessed_text]

processed_data = []
for doc in authors_corpus:
    tokens = preprocess_text(doc)
    processed_data.append(tokens)
    
gensim_dictionary = corpora.Dictionary(processed_data)
gensim_corpus = [gensim_dictionary.doc2bow(token, allow_update=True) for token in processed_data]

lda_model = gensim.models.ldamodel.LdaModel(gensim_corpus, num_topics=2, id2word=gensim_dictionary, passes=40)

topics = lda_model.print_topics(num_words=15)
for topic in topics:
    print(topic[1], '\n')



In [ ]:
# What topic each of the books seems to belong to
for g_corpus in gensim_corpus:
    print(lda_model[g_corpus])


In [ ]:
import pyLDAvis.gensim_models

lda_visualization = pyLDAvis.gensim_models.prepare(lda_model, gensim_corpus, gensim_dictionary, sort_topics=False)
pyLDAvis.display(lda_visualization)


## Exercise 2 - Visualization basics



Use the Jane Austen vs Dostoyevsky topic visualization
1. Inspect the conditional topic distribution given for the words 'elizabeth', 'sister' and 'suddenly'. (hover over the words to see their cond. topic distribution)


* elizabeth - only appears in Jane Austen
* sister - appears in both Jane Austen and Dostoyevsky, but more relevant in Jane Austen
* suddenly - appears significantly more in Dostoyevsky than in Jane Austen


2. Show the most relevant 30 terms of each topic when lambda = 0.



3. Answer the following questions based on visual observations:




>*   The term 'father' is more relevant for which author?
>*   In the books of which author do things happen 'suddenly' more often? 
>*   For which author does 'family' appear more often?
>*   In the books of which author do 'letter(s)' play a bigger role?



* father - more relavant in Jane Austen
* suddenly - significantly more in Dostoyevsky
* family - a lot more in Jane Austen
* letter(s) - more in Jane Austen

## Exercise 3 



Choose books from at least 3 different authors and create your clean corpus of books using lemmatization.
For one book of your choice (out of the >=3) split the book into chapters.


In [ ]:
books = [
    {'id': 996, 'name': 'Don Quixote'},
    {'id': 8486, 'name': 'Ghost Stories of an Antiquary'},
    {'id': 2600, 'name': 'War and Peace'},
    {'id': 164, 'name': 'Twenty Thousand Leagues under the Sea'}
]

def process_book(book_id):
    raw_book = gutenbergpy.textget.get_text_by_id(book_id)
    the_possessed_text = gutenbergpy.textget.strip_headers(raw_book).decode("utf-8")
    
    return the_possessed_text

author_content = [process_book(book['id']) for book in books]

In [ ]:
author_texts = []
for content in author_content:
    lemmatized_words = preprocess_text(content)
    author_texts.append(lemmatized_words)

In [ ]:
author_dict = corpora.Dictionary(author_texts)
author_corpus = [author_dict.doc2bow(text) for text in author_texts]


1. Train (use) an LDA Model that correctly separates everything into N distinct topics, where N is the number of authors you have. E.g. If you have 4 different authors your model should separate your corpus into 4 different topics.


In [ ]:
lda_model = gensim.models.ldamodel.LdaModel(author_corpus, num_topics=4, id2word=author_dict, passes=70)

topics = lda_model.print_topics(num_words=8)
for topic in topics:
    print(topic[1])


2. Separate the text into chapters for one book of your choice out of the ones chosen. Print the similitude of each chapter to all of the authors that you've chosen.
Your output should look like:


In [ ]:
import re

def split_into_chapters(book_text):
    return re.split(r'CHAPTER [IVXLCD]+\.', book_text)
    
chapters = split_into_chapters(author_content[0])

In [ ]:
for idx, chapter in enumerate(chapters):
    processeded_chapter = preprocess_text(chapter)
    bow_processeded_chapter = author_dict.doc2bow(processeded_chapter)
    
    print(f'Chpater {idx + 1}: {lda_model[bow_processeded_chapter]}')


    Chapter 1 - (Author 1)

    Chapter 2 - (Author 1 - 0.73, Author 3 - 0.25)

    Chapter 3 - (Author 1)

    etc.

    It's possible that a chapter is similar to the writing style or theme of multiple authors - in that case print out the names and similitude weights for the authors.


